# Multiprocessing in notebooks: compatibility investigation

Run all cells in a fresh kernel for each configuration. Set the environment
variable `QUTIP_NOTEBOOK_CONTEXT` before starting Jupyter to `default` (the
QuTiP default), `spawn`, `forkserver`, or `fork`. Each run compares multiprocessing
and loky backends with serial results. Backend failures are recorded as
observations so later cases can run; inspect the results and tracebacks. A kernel
crash or timeout must be recorded separately. Small workloads test correctness,
not performance.

`fork` may hang or crash a multithreaded process, particularly on macOS. A
successful run does not establish that it is safe. Use a disposable kernel.


In [ ]:
import importlib.metadata as metadata
import json
import multiprocessing as mp
import os
import platform
import subprocess
import sys
import time
import traceback

import IPython
import numpy as np
import qutip as qt

choice = os.environ.get('QUTIP_NOTEBOOK_CONTEXT', 'default')
available = mp.get_all_start_methods()
if choice != 'default':
    if choice not in available:
        raise RuntimeError(f'Start method {choice!r} unavailable; available: {available}')
    qt.settings.mp_context = choice

# Run after application-level set_start_method: get_context resolves the default.
configured = qt.settings.mp_context
application_default = mp.get_context().get_start_method()
effective = configured.get_start_method() if configured is not None else application_default
try:
    commit = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True,
                            text=True, check=True).stdout.strip()
except (OSError, subprocess.CalledProcessError):
    commit = os.environ.get('GITHUB_SHA', 'unknown')
environment = {
    'python': sys.version, 'platform': platform.platform(),
    'machine': platform.machine(), 'qutip': qt.__version__,
    'qutip_file': qt.__file__, 'commit': commit,
    'ipython': IPython.__version__, 'ipykernel': metadata.version('ipykernel'),
    'numpy': np.__version__, 'loky': metadata.version('loky'),
    'kernel_pid': os.getpid(), 'selection': choice,
    'qutip_context': configured.get_start_method() if configured else None,
    'application_default': application_default, 'effective_context': effective,
    'available_methods': available,
}
print(json.dumps(environment, indent=2))


## Test harness and serial baselines

Fixed seeds allow trajectory averages to be compared. Baseline failures stop
the notebook. Backend exceptions and comparison failures are recorded as
observations; the final summary reports them explicitly.


In [ ]:
WORKERS = 2
NTRAJ = 8
SEED = 1234
observations = []

def observe(case, backend, run, check):
    start = time.perf_counter()
    try:
        result = run()
        check(result)
    except Exception as exc:
        elapsed = time.perf_counter() - start
        observations.append({'case': case, 'backend': backend, 'status': 'FAIL',
                             'seconds': round(elapsed, 3),
                             'error': f'{type(exc).__name__}: {exc}'})
        print(f'FAIL {case} / {backend} ({elapsed:.3f}s)')
        traceback.print_exc()
    else:
        elapsed = time.perf_counter() - start
        observations.append({'case': case, 'backend': backend, 'status': 'PASS',
                             'seconds': round(elapsed, 3)})
        print(f'PASS {case} / {backend} ({elapsed:.3f}s)')

def compare(actual, expected):
    np.testing.assert_allclose(actual, expected, rtol=1e-7, atol=1e-7)

H = 0.5 * qt.sigmax()
psi0 = qt.basis(2, 0)
rho0 = qt.ket2dm(psi0)
times = np.linspace(0, 1, 11)
sm = qt.sigmam()
c_ops = [np.sqrt(0.2) * sm]
e_ops = [qt.sigmaz()]
serial_options = {'map': 'serial', 'progress_bar': False}
def parallel_options(backend):
    return {'map': backend, 'num_cpus': WORKERS, 'progress_bar': False}


## Cell-defined callback: `parallel_map` and `loky_pmap`

The callback and referenced value are defined in the notebook. Successful work
must run outside the kernel. Submitted work must be serialized even under fork.


In [ ]:
factor = 3
def cell_task(value):
    return factor * value, os.getpid()

values = list(range(8))
expected_map = [factor * value for value in values]
def check_map(result):
    assert [item[0] for item in result] == expected_map
    assert all(pid != os.getpid() for _, pid in result)
    print('Worker PIDs:', sorted({pid for _, pid in result}))

for backend, mapper in [('parallel', qt.parallel_map), ('loky', qt.loky_pmap)]:
    observe('cell-defined map callback', backend,
            lambda mapper=mapper: mapper(cell_task, values, map_kw={'num_cpus': WORKERS}),
            check_map)


## Monte Carlo trajectories

Compare seeded trajectories with serial.


In [ ]:
def run_mc(backend):
    options = serial_options if backend == 'serial' else parallel_options(backend)
    return qt.mcsolve(H, psi0, times, c_ops, e_ops=e_ops, ntraj=NTRAJ,
                      seeds=SEED, options=options).average_expect

mc_baseline = run_mc('serial')
for backend in ('parallel', 'loky'):
    observe('mcsolve', backend, lambda backend=backend: run_mc(backend),
            lambda actual: compare(actual, mc_baseline))


## Stochastic master-equation trajectories

Compare another trajectory solver with seeded serial results.


In [ ]:
def run_sme(backend):
    options = serial_options if backend == 'serial' else parallel_options(backend)
    return qt.smesolve(H, rho0, times, sc_ops=c_ops, e_ops=e_ops, ntraj=NTRAJ,
                       seeds=SEED, options=options).average_expect

sme_baseline = run_sme('serial')
for backend in ('parallel', 'loky'):
    observe('smesolve', backend, lambda backend=backend: run_sme(backend),
            lambda actual: compare(actual, sme_baseline))


## Non-Markovian Monte Carlo and stochastic Schrödinger trajectories

Compare two additional solver families with seeded serial results.


In [ ]:
def run_nm(backend):
    options = serial_options if backend == 'serial' else parallel_options(backend)
    return qt.nm_mcsolve(H, psi0, times, [(sm, 0.2)], e_ops=e_ops,
                         ntraj=NTRAJ, seeds=SEED, options=options).average_expect

def run_sse(backend):
    options = serial_options if backend == 'serial' else parallel_options(backend)
    return qt.ssesolve(H, psi0, times, sc_ops=c_ops, e_ops=e_ops, ntraj=NTRAJ,
                       seeds=SEED, options=options).average_expect

for case, run in [('nm_mcsolve', run_nm), ('ssesolve', run_sse)]:
    baseline = run('serial')
    for backend in ('parallel', 'loky'):
        observe(case, backend, lambda backend=backend: run(backend),
                lambda actual: compare(actual, baseline))


## Cell-defined time-dependent coefficient

This callback is part of the solver state.


In [ ]:
def drive(t, frequency):
    return np.cos(frequency * t)

driven_H = qt.QobjEvo([H, [0.1 * qt.sigmaz(), drive]], args={'frequency': 2.0})
def run_driven(backend):
    options = serial_options if backend == 'serial' else parallel_options(backend)
    return qt.mcsolve(driven_H, psi0, times, c_ops, e_ops=e_ops, ntraj=NTRAJ,
                      seeds=SEED, options=options).average_expect

driven_baseline = run_driven('serial')
for backend in ('parallel', 'loky'):
    observe('mcsolve cell-defined coefficient', backend,
            lambda backend=backend: run_driven(backend),
            lambda actual: compare(actual, driven_baseline))


## Correlation with a solver instance

This maps a calculation carrying a solver object.


In [ ]:
tlist = np.linspace(0, 1, 5)
taulist = np.linspace(0, 0.5, 6)
solver = qt.MESolver(H, c_ops, options={'progress_bar': False})
correlation_args = (solver, rho0, tlist, taulist, qt.sigmap(), qt.sigmaz(), sm)
correlation_baseline = qt.correlation_3op(*correlation_args, map='serial')
for backend in ('parallel', 'loky'):
    observe('correlation solver instance', backend,
            lambda backend=backend: qt.correlation_3op(
                *correlation_args, map=backend, map_kw={'num_cpus': WORKERS}),
            lambda actual: compare(actual, correlation_baseline))


## Results

Failures are compatibility findings, not automatically CI failures. Inspect
tracebacks and compare fresh-kernel runs. If this cell is never reached, inspect
the runner's timeout or kernel-failure result.


In [ ]:
print(json.dumps({'environment': environment, 'observations': observations}, indent=2))
print('Recorded', len(observations), 'observations;',
      sum(item['status'] == 'FAIL' for item in observations), 'failed')
